In [4]:
import pandas as pd
from src.f1_platform.db.connection import get_engine

engine = get_engine()
print("Connected to:", engine.url.database)

Connected to: f1_data


In [5]:
laps = pd.read_sql("SELECT * FROM bronze.raw_laps", engine)
results = pd.read_sql("SELECT * FROM bronze.raw_results", engine)
weather = pd.read_sql("SELECT * FROM bronze.raw_weather_data", engine)
telemetry = pd.read_sql("SELECT * FROM bronze.raw_telemetry", engine)

print(f"laps:      {len(laps):>8,} rows × {len(laps.columns):>3} cols")
print(f"results:   {len(results):>8,} rows × {len(results.columns):>3} cols")
print(f"weather:   {len(weather):>8,} rows × {len(weather.columns):>3} cols")
print(f"telemetry: {len(telemetry):>8,} rows × {len(telemetry.columns):>3} cols")

laps:        30,959 rows ×  35 cols
results:        565 rows ×  26 cols
weather:      4,398 rows ×  12 cols
telemetry:   18,155 rows ×  23 cols


In [6]:
laps.dtypes

Time                           int64
Driver                        object
DriverNumber                  object
LapTime                        int64
LapNumber                    float64
Stint                        float64
PitOutTime                     int64
PitInTime                      int64
Sector1Time                    int64
Sector2Time                    int64
Sector3Time                    int64
Sector1SessionTime             int64
Sector2SessionTime             int64
Sector3SessionTime             int64
SpeedI1                      float64
SpeedI2                      float64
SpeedFL                      float64
SpeedST                      float64
IsPersonalBest                object
Compound                      object
TyreLife                     float64
FreshTyre                       bool
Team                          object
LapStartTime                   int64
LapStartDate          datetime64[ns]
TrackStatus                   object
Position                     float64
D

## Bronze Laps — Type Analysis

### Issues to fix in Silver:

**Time columns stored as int64 (nanoseconds):**
- `Time`, `LapTime`, `PitOutTime`, `PitInTime`
- `Sector1Time`, `Sector2Time`, `Sector3Time`
- `Sector1SessionTime`, `Sector2SessionTime`, `Sector3SessionTime`
- `LapStartTime`
- → Convert to seconds (float)

**Wrong type:**
- `IsPersonalBest`: object → should be bool
- `DriverNumber`: object → consider int

**Sentinel values for missing data:**
- `-9223372036854775808` appears where NaN should be
- → Replace with NULL during Silver transformation

In [7]:
laps.isna().sum()

Time                    0
Driver                  0
DriverNumber            0
LapTime                 0
LapNumber               0
Stint                 446
PitOutTime              0
PitInTime               0
Sector1Time             0
Sector2Time             0
Sector3Time             0
Sector1SessionTime      0
Sector2SessionTime      0
Sector3SessionTime      0
SpeedI1                42
SpeedI2               927
SpeedFL               990
SpeedST                50
IsPersonalBest         23
Compound                0
TyreLife              526
FreshTyre               0
Team                    0
LapStartTime            0
LapStartDate            0
TrackStatus             0
Position               42
Deleted                 0
DeletedReason          23
FastF1Generated         0
IsAccurate              0
event                   0
year                    0
ingested_at             0
source                  0
dtype: int64

In [ ]:
sentinel = -9223372036854775808

# How many sentinel values we have for each int column?
print(f"LapTime sentinels:     {(laps['LapTime'] == sentinel).sum():>6,}")
print(f"PitOutTime sentinels:  {(laps['PitOutTime'] == sentinel).sum():>6,}")
print(f"PitInTime sentinels:   {(laps['PitInTime'] == sentinel).sum():>6,}")
print(f"Sector1Time sentinels: {(laps['Sector1Time'] == sentinel).sum():>6,}")
print(f"Sector2Time sentinels: {(laps['Sector2Time'] == sentinel).sum():>6,}")
print(f"Sector3Time sentinels: {(laps['Sector3Time'] == sentinel).sum():>6,}")

LapTime sentinels:        403
PitOutTime sentinels:  29,991
PitInTime sentinels:   30,011
Sector1Time sentinels:    608
Sector2Time sentinels:     62
Sector3Time sentinels:     68


In [10]:
print("=== Compound (tire types) ===")
print(laps['Compound'].value_counts())
print()
print("=== Number of unique drivers ===")
print(f"Total unique drivers: {laps['Driver'].nunique()}")
print()
print("=== Top 5 drivers by lap count ===")
print(laps['Driver'].value_counts())

=== Compound (tire types) ===
Compound
HARD            13278
MEDIUM          11905
SOFT             3676
INTERMEDIATE     1597
nan               446
None               57
Name: count, dtype: int64

=== Number of unique drivers ===
Total unique drivers: 24

=== Top 5 drivers by lap count ===
Driver
RUS    1671
OCO    1628
NOR    1608
VER    1593
HAM    1589
LEC    1588
BEA    1569
GAS    1542
ANT    1541
HAD    1525
PIA    1510
LAW    1503
ALB    1477
SAI    1474
STR    1461
HUL    1458
TSU    1447
ALO    1427
BOR    1405
COL    1275
DOO     217
LIN     165
PER     163
BOT     123
Name: count, dtype: int64


In [11]:
print("=== LapNumber range ===")
print(f"Min: {laps['LapNumber'].min()}")
print(f"Max: {laps['LapNumber'].max()}")
print()
print("=== Speed ranges (km/h) ===")
print(f"SpeedI1  range: {laps['SpeedI1'].min()} - {laps['SpeedI1'].max()}")
print(f"SpeedST  range: {laps['SpeedST'].min()} - {laps['SpeedST'].max()}")
print(f"SpeedFL  range: {laps['SpeedFL'].min()} - {laps['SpeedFL'].max()}")

=== LapNumber range ===
Min: 1.0
Max: 78.0

=== Speed ranges (km/h) ===
SpeedI1  range: 31.0 - 352.0
SpeedST  range: 54.0 - 364.0
SpeedFL  range: 62.0 - 359.0


In [16]:
sentinel = -9223372036854775808
valid_lap_times = laps[laps['LapTime'] != sentinel]['LapTime']
lap_times_seconds = valid_lap_times / 1_000_000_000

print(f"Valid lap times: {len(lap_times_seconds):,} from {len(laps):,} total")
print()
print(f"Fastest lap: {lap_times_seconds.min():.3f} seconds")
print(f"Slowest lap: {lap_times_seconds.max():.3f} seconds")
print(f"Median lap: {lap_times_seconds.median():.3f} seconds")
print(f"Mean lap: {lap_times_seconds.mean():.3f} seconds")

Valid lap times: 30,556 from 30,959 total

Fastest lap: 67.924 seconds
Slowest lap: 218.679 seconds
Median lap: 91.913 seconds
Mean lap: 91.626 seconds


# Data Quality Checks

Following standard ETL profiling practices before Silver layer transformation.

## Check 1: Duplicates